# 🧠 Multimodal Deep Learning Tutorial: Vision + Text Fusion Architecture

Welcome to this comprehensive tutorial on **Multimodal Deep Learning** using **PyTorch**, **NumPy**, and **Matplotlib**.

--- 
## 🔰 Beginner's Guide & Key Concepts

### What is Multimodal AI?
Single-modality AI models look at only one type of data (e.g. only images or only text). **Multimodal AI models** combine multiple data types—such as **Image Vectors** and **Text Embeddings**—into one neural network to make smarter decisions.

### Quick Beginner Concepts:
1. **Image Features**: Array of numbers extracted from a picture (e.g. dim 64).
2. **Text Embeddings**: Vector representation of words/sentences (e.g. dim 128).
3. **Feature Fusion**: Joining vectors using `torch.cat((image_features, text_embeddings), dim=1)` so PyTorch learns from both simultaneously.

---

## 🚀 Simple Quickstart Implementation (Beginner 5-Line Fusion Code)
Here is how multimodal feature fusion works in PyTorch in 5 simple lines:

In [ ]:
# === SIMPLE BEGINNER IMPLEMENTATION ===
import torch

image_vector = torch.randn(1, 64)    # 1 image sample with 64 features
text_vector = torch.randn(1, 128)    # 1 sentence embedding with 128 features

# Join (Fuse) image + text features into a single combined vector
fused_vector = torch.cat((image_vector, text_vector), dim=1)

print(f"Simple Fusion Completed! Joined Vector Shape: {fused_vector.shape}") # Shape will be [1, 192]

--- 
## 🛠️ Step 1: Import Libraries & Generate Multimodal Synthetic Dataset
We create a synthetic dataset consisting of paired **Image Feature Vectors** (from a simulated CNN/Vision Transformer) and **Text Embeddings** (from a simulated BERT/Language Transformer).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Set random seeds so everyone gets identical results
torch.manual_seed(42)
np.random.seed(42)

# Custom PyTorch Dataset class for Multimodal samples
class MultimodalSyntheticDataset(Dataset):
    def __init__(self, num_samples=500, img_dim=64, text_dim=128):
        self.img_data = torch.randn(num_samples, img_dim)    # 500 image feature vectors
        self.text_data = torch.randn(num_samples, text_dim)  # 500 text embedding vectors
        
        # Create target labels (0 or 1) based on combined image + text interaction
        interaction = (self.img_data[:, :5].sum(dim=1) + self.text_data[:, :5].sum(dim=1))
        self.labels = (interaction > 0).long()
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return self.img_data[idx], self.text_data[idx], self.labels[idx]

# Instantiate DataLoader with mini-batches of size 32
dataset = MultimodalSyntheticDataset()
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset created with {len(dataset)} multimodal samples.")
img_sample, text_sample, label_sample = dataset[0]
print(f"Sample Image Shape: {img_sample.shape}, Sample Text Shape: {text_sample.shape}, Label: {label_sample.item()}")

--- 
## 🔹 Step 2: Build Dual-Stream Multimodal Fusion Network Architecture

We construct a PyTorch module with:
1. **Vision Sub-Network:** Processes image spatial vectors.
2. **Text Sub-Network:** Processes textual embedding vectors.
3. **Fusion Layer:** Concatenates both modality embeddings and passes through a Joint Multimodal Classifier.

In [ ]:
class MultimodalFusionNet(nn.Module):
    def __init__(self, img_dim=64, text_dim=128, hidden_dim=64, num_classes=2):
        super(MultimodalFusionNet, self).__init__()
        
        # Stream 1: Image Encoder
        self.img_encoder = nn.Sequential(
            nn.Linear(img_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Stream 2: Text Encoder
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Joint Multimodal Fusion Classifier (Hidden_dim * 2 = 128 -> 2 classes)
        self.fusion_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, img_x, text_x):
        # Pass image and text through respective encoders
        img_feat = self.img_encoder(img_x)
        text_feat = self.text_encoder(text_x)
        
        # Fuse (Concatenate) features along column dimension
        fused_features = torch.cat((img_feat, text_feat), dim=1)
        
        # Get final classification prediction
        output = self.fusion_classifier(fused_features)
        return output

model = MultimodalFusionNet()
print(model)

--- 
## 🔹 Step 3: Train the Multimodal Neural Network

We use Cross-Entropy Loss and Adam Optimizer to train the joint fusion model over 20 epochs.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 20
loss_history = []
acc_history = []

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for imgs, texts, labels in train_loader:
        optimizer.zero_grad()               # Reset gradients
        outputs = model(imgs, texts)        # Forward pass (Image + Text)
        loss = criterion(outputs, labels)   # Calculate loss
        loss.backward()                     # Backward pass
        optimizer.step()                    # Update model weights
        
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    loss_history.append(epoch_loss)
    acc_history.append(epoch_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%")

--- 
## 🔹 Step 4: Visualize Training Curves & Performance

Plot Loss convergence and Accuracy metrics during multimodal joint training.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(range(1, epochs+1), loss_history, 'r-o', linewidth=2)
ax1.set_title("Multimodal Training Loss Convergence")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True)

# Accuracy plot
ax2.plot(range(1, epochs+1), [a * 100 for a in acc_history], 'g-s', linewidth=2)
ax2.set_title("Multimodal Classification Accuracy (%)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True)

plt.tight_layout()
plt.show()

--- 
## 🎯 Summary for Beginners

- Multimodal AI combines features from two or more modalities.
- In PyTorch, use separate Linear/Encoder blocks for images and text.
- Merge them using `torch.cat((img_features, text_features), dim=1)` before sending to the classifier layer.